In [1]:
import torch
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

Torch: 2.12.0.dev20260217+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 5080


In [2]:
import numpy as np

In [ ]:
try:
    from bels_project_attempt.device_segmentation_gui_app import create_device_segmentation_app
except ModuleNotFoundError:
    from device_segmentation_gui_app import create_device_segmentation_app

app = create_device_segmentation_app()
viewer = app.viewer
segment_and_view = app.segment_and_view
list_images = app.list_images
images_output = app.images_output
main_panel = app.main_panel
get_cropped_outputs = app.get_cropped_outputs
get_z_tracker = app.get_z_tracker

def map_stage_z_to_original(stage_name: str, z_index: int):
    return app.z_to_original(stage_name, z_index)

def print_z_tracker():
    print(app.z_tracker_summary())

cropped_xyz = app.cropped_xyz
print("GUI ready. cropped_xyz:", None if cropped_xyz is None else cropped_xyz.shape)

GUI ready. cropped_xyz: None


c:\Users\taylorhearn\git_repos\vascumap\bels_project_attempt\device_segmentation_gui_app.py:898: FutureWarning: `estimate` is deprecated since version 0.26 and will be removed in version 2.2. Please use `ProjectiveTransform.from_estimate` class constructor instead.
  if not tform.estimate(dst, c):
c:\Users\taylorhearn\git_repos\vascumap\bels_project_attempt\device_segmentation_gui_app.py:938: FutureWarning: `estimate` is deprecated since version 0.26 and will be removed in version 2.2. Please use `ProjectiveTransform.from_estimate` class constructor instead.
  if not tform.estimate(dst, c):


In [5]:
required_symbols = ["app", "np"]
missing = [name for name in required_symbols if name not in globals()]
if missing:
    raise RuntimeError(f"Missing required symbols: {missing}. Run Cell 3 first.")

def _lock_z_controls(gui_app):
    try:
        gui_app.crop_z.call_button.enabled = False
        gui_app.crop_z.z_range_um.enabled = False
        gui_app.crop_z.z_min.enabled = False
        gui_app.crop_z.z_max.enabled = False
    except Exception:
        pass

_lock_z_controls(app)

if not hasattr(app, "_orig_apply_crop_from_roi"):
    app._orig_apply_crop_from_roi = app._apply_crop_from_roi

    def _apply_crop_with_auto_z_and_debug():
        app._orig_apply_crop_from_roi()
        _lock_z_controls(app)

        active_stack = app._get_active_stack_for_z()
        if active_stack is None:
            return

        z_min, z_max, dbg_txt = app._compute_vote_driven_z_bounds(min_total_um=160.0, min_down_um=30.0)
        if z_min is None or z_max is None:
            app.images_output.value = f"{app.images_output.value}\n[DEBUG] Unable to choose z slices automatically using vote>1 contiguous rule. {dbg_txt}"
            return

        app._apply_z_crop()

        counts = app._last_geometry_vote_counts
        if counts is None:
            voted_txt = "none"
        else:
            voted_idx = np.flatnonzero(np.asarray(counts) > 1).astype(int).tolist()
            voted_txt = ",".join(str(v) for v in voted_idx) if voted_idx else "none"

        out_nz = 0 if app.cropped_xyz is None else int(np.asarray(app.cropped_xyz).shape[0])
        debug_line = (
            f"[DEBUG] voted_slices_vote_gt_1={voted_txt}; chosen_slices={int(z_min)}..{int(z_max)}; output_stack_height={out_nz}"
        )
        app.images_output.value = f"{app.images_output.value}\n{debug_line}\n{dbg_txt}"

    app._apply_crop_from_roi = _apply_crop_with_auto_z_and_debug

def export_gui_cropped_stack(gui_app) -> tuple[np.ndarray, tuple[float, float, float]]:
    if gui_app is None or getattr(gui_app, "cropped_xyz", None) is None:
        raise RuntimeError("No GUI cropped stack found. Run GUI crop first.")

    cropped = np.asarray(gui_app.cropped_xyz, dtype=np.float32).copy()
    lz, ly, lx = getattr(gui_app, "_loaded_voxel_um", (None, None, None))
    z_um = lz if lz is not None else getattr(gui_app, "_last_z_step_um", None)
    y_um = ly if ly is not None else (getattr(gui_app, "_last_y_step_um", None) or getattr(gui_app, "_last_xy_step_um", None))
    x_um = lx if lx is not None else (getattr(gui_app, "_last_x_step_um", None) or getattr(gui_app, "_last_xy_step_um", None))

    if z_um is None or y_um is None or x_um is None:
        raise RuntimeError("Missing voxel metadata on GUI app.")

    src_voxel_um = (float(z_um), float(y_um), float(x_um))
    return cropped, src_voxel_um

print("GUI is configured for automatic z-selection debug output (vote>1 contiguous rule).")
print("Run Cell 3 GUI, then click: Segment + View -> Create cropped aligned")

GUI is configured for automatic z-selection debug output (vote>1 contiguous rule).
Run Cell 3 GUI, then click: Segment + View -> Create cropped aligned


In [6]:
from pathlib import Path
try:
    from bels_project_attempt.translation_segmentation_pipeline import (
        default_checkpoints,
        run_translation_and_segmentation,
    )
except ModuleNotFoundError:
    from translation_segmentation_pipeline import (
        default_checkpoints,
        run_translation_and_segmentation,
    )

if "app" not in globals() or app is None:
    raise RuntimeError("Missing GUI app. Run Cell 3 first.")

cropped_z_stack, src_voxel_um = export_gui_cropped_stack(app)
pix2pix_model_path, seg_model_path = default_checkpoints(Path.cwd())
device_width_um = float(getattr(app, "_active_device_width_um", 30.0))

translated_image, segmentation_probability_map, segmentation_mask = run_translation_and_segmentation(
    cropped_z_stack=cropped_z_stack,
    src_voxel_um=src_voxel_um,
    pix2pix_model_path=pix2pix_model_path,
    seg_model_path=seg_model_path,
    device="cuda",
    n_iter=1,
    ref_voxel_um=(5.0, 2.0, 2.0),
    iso_voxel_um=(2.0, 2.0, 2.0),
    device_width_um=device_width_um,
    use_gpu=True,
)

print("Translation + segmentation complete")
print(f"  device_width_removed_per_side_um: {device_width_um:.4g}")
print("  image_translation:", tuple(int(v) for v in np.asarray(translated_image).shape[:3]))
print("  segmentation_probability_map:", tuple(int(v) for v in np.asarray(segmentation_probability_map).shape[:3]))
print("  segmentation_mask:", tuple(int(v) for v in np.asarray(segmentation_mask).shape[:3]))

c:\Users\taylorhearn\AppData\Local\miniconda3\envs\clean_vascumap\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\taylorhearn\AppData\Local\miniconda3\envs\clean_vascumap\Lib\site-packages\monai\networks\nets\unet.py:130: UserWarning: `len(strides) > len(channels) - 1`, the last 1 values of strides will not be used.
  warnings.warn(f"`len(strides) > len(channels) - 1`, the last {delta} values of strides will not be used.")
  0%|          | 0/48 [00:00<?, ?it/s]c:\Users\taylorhearn\AppData\Local\miniconda3\envs\clean_vascumap\Lib\site-packages\monai\inferers\utils.py:231: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], wh

Translation + segmentation complete
  device_width_removed_per_side_um: 40
  image_translation: (85, 2387, 1479)
  segmentation_probability_map: (85, 2387, 1479)
  segmentation_mask: (85, 2387, 1479)


In [7]:
import napari
viewer = napari.Viewer()
viewer.add_image(translated_image, name="translated_image")
viewer.add_image(segmentation_probability_map, name="seg")
viewer.add_labels(  segmentation_mask.astype(np.uint8), name="segmentation_mask")

<Labels layer 'segmentation_mask' at 0x29848104610>